In [3]:
from cellgrn.main import summarize_grn,calc_cellgrn_top_target_score,calc_cellgrn_top_target_connectivity
from cellgrn.utils import grn_umap
import numpy as np
import pandas as pd
import os
import anndata as ad
import pickle
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import sparse

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [1]:
# def calc_cellgrn_top_target_score(df_grn, cand_df, gene_list, sample_grn_res,
#                                 col_name='TF', ntop=500):
#     out_df = pd.DataFrame()
#     for test_tf in gene_list:
#         sel_grn  =cand_df[cand_df[col_name]==test_tf]
#         sel_names = sel_grn.apply(lambda row: "_".join(row.astype(str)), axis=1).tolist()
#         sel_df = df_grn.copy()[sel_names]
#         sel_names2 = sample_grn_res.loc[sel_names,:].nlargest(ntop, "score").index.values 
#         sel_df2 = sel_df[sel_names2].sum(axis=1)
#         out_df = pd.concat([out_df,sel_df2],axis=1)
#     out_df.columns = gene_list
#     return out_df 

# def calc_cellgrn_top_target_connectivity(df_grn, cand_df, gene_list, 
#                                 col_name='TF', rank=5000):
#     out_df = pd.DataFrame()
#     for test_tf in gene_list:
#         sel_grn  =cand_df[cand_df[col_name]==test_tf]
#         sel_names = sel_grn.apply(lambda row: "_".join(row.astype(str)), axis=1).tolist()
#         sel_df = df_grn.copy()[sel_names]
#         df_row_ranks = df_grn.rank(axis=1, ascending=False, method="first")
#         selected_ranks = df_row_ranks[sel_names]
#         sel_df2 = ((sel_df > 0) & (selected_ranks <= rank)).sum(axis=1)
#         out_df = pd.concat([out_df,sel_df2],axis=1)
#     out_df.columns = gene_list
#     return out_df 

In [5]:
cell_meta = pd.read_csv("/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/metadata.csv",header=0)
cell_types = cell_meta['Sample']

with open("/home/shaliu_fu/multireg/cellGRN/output/res_spatial_brain_scenic2/scenic2_cell_grn.pkl", "rb") as f:
    grn_scale2 = pickle.load(f)

In [6]:


sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)


In [7]:
sel_tf = ['Pax6','Eomes','Tbr1','Neurod1']

cand_df_tf_peak = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/output/res_spatial_brain_scenic2/tf_peak_celltype_scale2.csv",header=0)
cand_df_tf_peak = cand_df_tf_peak[cand_df_tf_peak['TF'].isin(sel_tf)]
cand_df_tf_peak['TF'].value_counts() # scenic没预测到。



TF
Pax6    73
Name: count, dtype: int64

In [8]:
cand_df_tf_peak2 = cand_df_tf_peak.copy()
cand_df_tf_peak2['group_subtype'] = "TF-Peak"
cand_df_tf_peak2 = cand_df_tf_peak2[['group_subtype',"TF","Peak"]]

In [9]:
tf_peak_regulon = calc_cellgrn_top_target_score(grn_scale2.copy(),cand_df_tf_peak2.copy(),sel_tf,sample_grn_scale2,col_name='TF',ntop=1000)

In [10]:
cand_df_tf_gene =  pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/output/res_spatial_brain_scenic2/tf_gene_celltype_scale2.csv",header=0)

cand_df_tf_gene = cand_df_tf_gene[cand_df_tf_gene['TF'].isin(sel_tf)]
cand_df_tf_gene['TF'].value_counts()

TF
Pax6    23
Name: count, dtype: int64

In [11]:
cand_df_tf_gene2 = cand_df_tf_gene.copy()
cand_df_tf_gene2['group_subtype'] = "TF-Gene"
cand_df_tf_gene2 = cand_df_tf_gene2[['group_subtype',"TF","Gene"]]

In [12]:
tf_gene_regulon = calc_cellgrn_top_target_score(grn_scale2.copy(),cand_df_tf_gene2.copy(),sel_tf,sample_grn_scale2,ntop=500)

In [25]:
tf_gene_regulon.to_csv("/home/shaliu_fu/multireg/cellGRN/eval/results/spatial_brain/spatial_brain_tf_gene_activity.csv",header=True,index=True)
tf_peak_regulon.to_csv("/home/shaliu_fu/multireg/cellGRN/eval/results/spatial_brain/spatial_brain_tf_peak_activity.csv",header=True,index=True)


In [13]:
rna_expr = pd.read_csv("/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/rna_count.csv",header=0,index_col=0)

In [14]:
sel_tf_expr = rna_expr.loc[sel_tf,:].T

In [28]:
sel_tf_expr.to_csv("/home/shaliu_fu/multireg/cellGRN/eval/results/spatial_brain/sel_tf_expr.csv",header=True,index=True)

In [15]:
cand_df_gene_peak = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/output/res_spatial_brain_scenic2/gene_peak_sample_scale2.csv",header=0)
cand_df_gene_peak = cand_df_gene_peak[cand_df_gene_peak['Gene']=="Eomes"]
# cand_df_gene_peak['Gene'].value_counts() # scenic没预测到。


In [36]:
cand_df_gene_peak2 = cand_df_gene_peak.copy()
cand_df_gene_peak2['group_subtype'] = "Peak-Gene"
cand_df_gene_peak2 = cand_df_gene_peak2[['group_subtype',"Peak","Gene"]]

In [56]:
gene_peak_regulon = calc_cellgrn_top_target_score(grn_scale2.copy(),cand_df_gene_peak2.copy(),['Eomes'],sample_grn_scale2,col_name='Gene',ntop=1000)

In [ ]:
gene_peak_conn = calc_cellgrn_top_target_connectivity(grn_scale2.copy(),cand_df_gene_peak2.copy(),['Eomes'],col_name='Gene',rank=20000)

In [19]:
sel_gene_expr = rna_expr.loc[['Eomes'],:].T

In [50]:
# sel_gene_expr[sel_gene_expr['Eomes']>0]

In [49]:
# gene_peak_regulon[gene_peak_regulon['Eomes']>0]

In [ ]:
# def calc_cellgrn_top_target_score(df_grn, cand_df, gene_list, sample_grn_res,
#                                 col_name='TF', ntop=500):
#     out_df = pd.DataFrame()
#     for test_tf in gene_list:
#         sel_grn  =cand_df[cand_df[col_name]==test_tf]
#         sel_names = sel_grn.apply(lambda row: "_".join(row.astype(str)), axis=1).tolist()
#         sel_df = df_grn.copy()[sel_names]
#         sel_names2 = sample_grn_res.loc[sel_names,:].nlargest(ntop, "score").index.values 
#         sel_df2 = sel_df[sel_names2].sum(axis=1)
#         out_df = pd.concat([out_df,sel_df2],axis=1)
#     out_df.columns = gene_list
#     return out_df 

In [51]:
# sel_names = cand_df_gene_peak2.apply(lambda row: "_".join(row.astype(str)), axis=1).tolist()

In [ ]:
# sel_names

['Peak-Gene_chr9:118455364-118455864_Eomes',
 'Peak-Gene_chr9:118228695-118229195_Eomes',
 'Peak-Gene_chr9:118318700-118319200_Eomes',
 'Peak-Gene_chr9:118246831-118247331_Eomes']

In [52]:
# sample_grn_scale2.loc[sel_names,:]

In [66]:
out_tab = pd.concat([gene_peak_regulon,gene_peak_conn,sel_gene_expr],axis=1)

In [68]:
out_tab.columns = ["score","degree","expression"]
out_tab.to_csv("/home/shaliu_fu/multireg/cellGRN/eval/results/spatial_brain/spatial_brain_eomes_gene_peak_activity.csv")